In [1]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import sympy

## despues todo esto se pasara a un archivo .py para que se pueda importar y usar en otros scripts
#holi

In [2]:
def v_1(params, C):
    '''
    Kinetic Expression of the Phosphotransferase system
    Information
    Enzyme: PtsI, PtsH
    Reaction: glc + pep <-> g6p + pyr
    ---
    Parameters:

    params: dict
        Kinetic parameters of the model, including inhibition and activation constants,
        maximum reaction rates, and Hill coefficients.
    C: dict
        Concentrations of the relevant metabolites,
        such as pyruvate (C_pyr), phosphoenolpyruvate (C_pep) and glucose (C_glc) .
    Returns:

    v_1: float
        Flux expression for the phosphotransferase system,
        which is a function of the kinetic parameters and metabolite concentrations.

    '''
    # first term
    # primer _ es para los subíndices despues del
    # segundo _ es para los superíndices despues del
    k_pyr1_I = params['k_pyr1_I']
    # concentrations
    C_pyr = C['C_pyr']
    v_1 = (k_pyr1_I)/(k_pyr1_I + C_pyr) # <- inhibition term
    # second term
    k_pep1_A = params['k_pep1_A']
    C_pep = C['C_pep']
    v_1 += C_pep / (C_pep + k_pep1_A) # <- activation term

    # third term
    # en este caso, se divide la definicion del
    # numerador y el denominador para facilitar la lectura
    # numerador
    v_pts_max = params['v_pts_max']
    C_glc = C['C_glc']
    num = v_pts_max * C_glc * (C_pep/C_pyr)
    # denominador
    # de nuevo se divide en partes para facilitar la lectura
    # den1
    K_a1 = params['K_a1']
    K_a2 = params['K_a2']
    K_a3 = params['K_a3']
    den1 = K_a1 + K_a2 * (C_pep/C_pyr) + K_a3 * (C_glc) + C_glc * (C_pep/C_pyr)

    # den2
    n_g6p = params['n_g6p']
    K_g6p = params['K_g6p']
    den2 = 1 + (C_glc**n_g6p) / K_g6p
    v_1 += num / (den1 * den2)
    return v_1

# notar que aca usamos sympy unicamente para mostrar la expresion simbolica de la velocidad,
# cuando se trabaje de rial se usaran los valores numericos de los parametros y las concentraciones.
k_pyr1_I, k_pep1_A, v_pts_max, K_a1, K_a2, K_a3, n_g6p, K_g6p = sympy.symbols('k_pyr1_I k_pep1_A v_pts_max K_a1 K_a2 K_a3 n_g6p K_g6p')
C_pyr, C_pep, C_glc = sympy.symbols('C_pyr C_pep C_glc')
# parametros
params = {'k_pyr1_I': k_pyr1_I,
          'k_pep1_A': k_pep1_A,
          'v_pts_max': v_pts_max,
          'K_a1': K_a1,
          'K_a2': K_a2,
          'K_a3': K_a3,
          'n_g6p': n_g6p,
          'K_g6p': K_g6p}
# concentraciones de metabolitos
C = {'C_pyr': C_pyr,
     'C_pep': C_pep,
     'C_glc': C_glc}
v_1_sym = v_1(params, C)
display(v_1_sym)


C_glc*C_pep*v_pts_max/(C_pyr*(C_glc**n_g6p/K_g6p + 1)*(C_glc*C_pep/C_pyr + C_glc*K_a3 + C_pep*K_a2/C_pyr + K_a1)) + C_pep/(C_pep + k_pep1_A) + k_pyr1_I/(C_pyr + k_pyr1_I)

In [3]:
# las concentraciones isgueintes son del tipo
def v_2(params, C):
    '''
    Kinetic Expression of the Glucose 6-phosphate isomerase
    Information
    Enzyme: Pgi
    Reaction: g6p <-> f6p
    ---
    Parameters:

    params: dict
        Kinetic parameters of the model, including inhibition and activation constants,
        maximum reaction rates, and Hill coefficients.
    C: dict
        Concentrations of the relevant metabolites,
        such as C_g6p, C_f6p and C_pep.
    e: dict
        Enzyme concentrations for Pgi.
    Returns:

    v_2: float
        Flux expression for the pyruvate kinase,
        which is a function of the kinetic parameters and metabolite concentrations.

    Glucose 6-phosphate isomerase (PGI)
    Reaction: g6p <-> f6p
    '''

    # inhibition term (pep)
    k_pep2_I = params['k_pep2_I']
    C_pep = C['C_pep']
    v_2 = k_pep2_I / (k_pep2_I + C_pep)

    # numerator
    Pgi = params['Pgi']
    kcat_f2 = params['kcat_f2']
    kcat_r2 = params['kcat_r2']
    C_g6p = C['C_g6p']
    C_f6p = C['C_f6p']
    Km_g6p_2 = params['Km_g6p_2']
    Km_f6p_2 = params['Km_f6p_2']

    num1 = kcat_f2 * (C_g6p / Km_g6p_2) - kcat_r2 * (C_f6p / Km_f6p_2)


    # denominator
    den1 = ((C_g6p / Km_g6p_2 + 1) + (C_f6p / Km_f6p_2 + 1) - 1)

    v_2 *= Pgi * num1 / den1
    return v_2
k_pep2_I,Km_g6p_2,Km_f6p_2, kcat_f2, kcat_r2,Pgi = sympy.symbols('k_pep2_I Km_g6p_2 Km_f6p_2 kcat_f_2 kcat_r_2 Pgi')
C_g6p, C_f6p, C_pep = sympy.symbols('C_g6p C_f6p C_pep')
# parametros
params = { 'k_pep2_I': k_pep2_I,
          'C_g6p': C_g6p,
          'Km_g6p_2': Km_g6p_2,
          'Km_f6p_2': Km_f6p_2,
          'kcat_f2':kcat_f2,
          'kcat_r2': kcat_r2,
           'Pgi':Pgi}
# concentraciones de metabolitos
C = {'C_g6p': C_g6p,
     'C_pep': C_pep,
     'C_f6p': C_f6p}
v_2_sym = v_2(params, C)
display(v_2_sym)


Pgi*k_pep2_I*(-C_f6p*kcat_r_2/Km_f6p_2 + C_g6p*kcat_f_2/Km_g6p_2)/((C_pep + k_pep2_I)*(C_f6p/Km_f6p_2 + C_g6p/Km_g6p_2 + 1))

In [4]:
def v_3(params, C):
    '''
    Phosphofructokinase (PFK)
    Reaction: f6p + atp <-> fbp + adp
    '''

    v_3 = 1

    # inhibition terms
    inhibitors = ['f6p','fbp','gtp','pep','pi']
    for met in inhibitors:
        kI = params[f'kI_{met}_3']
        C_met = C[f'C_{met}']
        v_3 *= kI / (kI + C_met)

    # activation terms
    for met in ['adp','gdp']:
        kA = params[f'kA_{met}_3']
        C_met = C[f'C_{met}']
        v_3 *= C_met / (kA + C_met)

    # kinetic term
    PfkB = params['PfkB']
    kcat_f3 = params['kcat_f_3']
    kcat_r3 = params['kcat_r_3']

    C_f6p = C['C_f6p']
    C_atp = C['C_atp']
    C_fbp = C['C_fbp']
    C_adp = C['C_adp']

    Km_f6p_3 = params['Km_f6p_3']
    Km_atp_3 = params['Km_atp_3']
    Km_fbp_3 = params['Km_fbp_3']
    Km_adp_3 = params['Km_adp_3']

    num1 = kcat_f3 * (C_f6p/Km_f6p_3) * (C_atp/Km_atp_3) - \
          kcat_r3 * (C_fbp/Km_fbp_3) * (C_adp/Km_adp_3)


    den = (C_f6p/Km_f6p_3 + 1)*(C_atp/Km_atp_3 + 1) + \
          (C_fbp/Km_fbp_3 + 1)*(C_adp/Km_adp_3 + 1) - 1

    v_3 *= PfkB * num1 / den
    return v_3
kI_f6p_3,kI_fbp_3,kI_gtp_3,kI_pep_3,kI_pi_3,=sympy.symbols('kI_f6p_3 kI_fbp_3 kI_gtp_3 kI_pep_3 kI_pi_3')
kA_adp_3,kA_gdp_3=sympy.symbols('kA_adp_3 kA_gdp_3')
Km_f6p_3,Km_atp_3,Km_fbp_3,Km_adp_3,kcat_f_3,kcat_r_3,PfkB= sympy.symbols('Km_f6p_3 Km_atp_3 Km_fbp_3 Km_adp_3 kcat_f_3 kcat_r_3 PfkB')

C_f6p, C_atp, C_fbp, C_adp, C_gtp, C_gdp, C_pep,C_pi= sympy.symbols('C_f6p C_atp C_fbp C_adp C_gtp C_gdp C_pep C_pi')

# parametros
params = { 'kcat_f_3':kcat_f_3,
          'kcat_r_3': kcat_r_3,
          'PfkB':PfkB, 'Km_f6p_3': Km_f6p_3,
          'Km_atp_3': Km_atp_3,
          'Km_fbp_3': Km_fbp_3,
          'Km_adp_3': Km_adp_3, 'kI_f6p_3': kI_f6p_3,'kI_fbp_3':kI_fbp_3, 'kI_gtp_3':kI_gtp_3, 'kI_pep_3':kI_pep_3, 'kI_pi_3':kI_pi_3,
          'kA_adp_3':kA_adp_3,'kA_gdp_3':kA_gdp_3}
# concentraciones de metabolitos
C = {'C_f6p': C_f6p,'C_atp': C_atp,'C_fbp': C_fbp,'C_adp': C_adp, 'C_atp':C_atp, 'C_gtp':C_gtp, 'C_pep':C_pep, 'C_pi':C_pi, 'C_gdp':C_gdp}
v_3_sym = v_3(params, C)
display(v_3_sym)

C_adp*C_gdp*PfkB*kI_f6p_3*kI_fbp_3*kI_gtp_3*kI_pep_3*kI_pi_3*(-C_adp*C_fbp*kcat_r_3/(Km_adp_3*Km_fbp_3) + C_atp*C_f6p*kcat_f_3/(Km_atp_3*Km_f6p_3))/((C_adp + kA_adp_3)*(C_f6p + kI_f6p_3)*(C_fbp + kI_fbp_3)*(C_gdp + kA_gdp_3)*(C_gtp + kI_gtp_3)*(C_pep + kI_pep_3)*(C_pi + kI_pi_3)*((C_adp/Km_adp_3 + 1)*(C_fbp/Km_fbp_3 + 1) + (C_atp/Km_atp_3 + 1)*(C_f6p/Km_f6p_3 + 1) - 1))

In [6]:
def make_symbols(param_names, conc_names):
    params_sym = sympy.symbols(' '.join(param_names))
    conc_sym = sympy.symbols(' '.join(conc_names))

    params = dict(zip(param_names, params_sym))
    C = dict(zip(conc_names, conc_sym))

    return params, C

In [7]:
def v_4(params, C):

    v = 1

    # inhibition
    for met in ['3pg','dhap','g3p']:
        v *= params[f'kI_{met}_4'] / (params[f'kI_{met}_4'] + C[f'C_{met}'])

    # activation
    v *= C['C_pep'] / (params['kA_pep_4'] + C['C_pep'])

    # kinetic
    num = params['kcat_f_4']*(C['C_fbp']/params['Km_fbp_4']) - \
          params['kcat_r_4']*(C['C_g3p']/params['Km_g3p_4'])*(C['C_dhap']/params['Km_dhap_4'])

    den = (C['C_fbp']/params['Km_fbp_4'] + 1) + \
          (C['C_g3p']/params['Km_g3p_4'] + 1)*(C['C_dhap']/params['Km_dhap_4'] + 1) - 1

    return v * params['FbaA'] * num / den


param_names = ['kI_3pg_4','kI_dhap_4','kI_g3p_4','kA_pep_4',
               'kcat_f_4','kcat_r_4','Km_fbp_4','Km_g3p_4','Km_dhap_4','FbaA']

conc_names = ['C_3pg','C_dhap','C_g3p','C_pep','C_fbp']

params, C = make_symbols(param_names, conc_names)

display(v_4(params, C))

C_pep*FbaA*kI_3pg_4*kI_dhap_4*kI_g3p_4*(-C_dhap*C_g3p*kcat_r_4/(Km_dhap_4*Km_g3p_4) + C_fbp*kcat_f_4/Km_fbp_4)/((C_3pg + kI_3pg_4)*(C_dhap + kI_dhap_4)*(C_g3p + kI_g3p_4)*(C_pep + kA_pep_4)*(C_fbp/Km_fbp_4 + (C_dhap/Km_dhap_4 + 1)*(C_g3p/Km_g3p_4 + 1)))

In [8]:
def v_5(params, C):

    num = params['kcat_f_5']*(C['C_dhap']/params['Km_dhap_5']) - \
          params['kcat_r_5']*(C['C_g3p']/params['Km_g3p_5'])

    den = (C['C_dhap']/params['Km_dhap_5'] + 1) + \
          (C['C_g3p']/params['Km_g3p_5'] + 1) - 1

    return params['TpiA'] * num / den


param_names = ['kcat_f_5','kcat_r_5','Km_dhap_5','Km_g3p_5','TpiA']
conc_names = ['C_dhap','C_g3p']

params, C = make_symbols(param_names, conc_names)

display(v_5(params, C))

TpiA*(C_dhap*kcat_f_5/Km_dhap_5 - C_g3p*kcat_r_5/Km_g3p_5)/(C_dhap/Km_dhap_5 + C_g3p/Km_g3p_5 + 1)

In [9]:
def v_6(params, C):

    v = 1

    # inhibition
    for met in ['adp','amp','atp']:
        v *= params[f'kI_{met}_6'] / (params[f'kI_{met}_6'] + C[f'C_{met}'])

    # kinetic
    num = params['kcat_f_6']*(C['C_g3p']/params['Km_g3p_6']) * \
          (C['C_pi']/params['Km_pi_6']) * \
          (C['C_nad']/params['Km_nad_6']) - \
          params['kcat_r_6']*(C['C_pgp']/params['Km_pgp_6']) * \
          (C['C_nadh']/params['Km_nadh_6'])

    den = (C['C_g3p']/params['Km_g3p_6'] + 1) * \
          (C['C_pi']/params['Km_pi_6'] + 1) * \
          (C['C_nad']/params['Km_nad_6'] + 1) + \
          (C['C_pgp']/params['Km_pgp_6'] + 1) * \
          (C['C_nadh']/params['Km_nadh_6'] + 1) - 1

    return v * params['GapA'] * num / den


param_names = ['kI_adp_6','kI_amp_6','kI_atp_6',
               'kcat_f_6','kcat_r_6',
               'Km_g3p_6','Km_pi_6','Km_nad_6',
               'Km_pgp_6','Km_nadh_6','GapA']

conc_names = ['C_adp','C_amp','C_atp',
              'C_g3p','C_pi','C_nad',
              'C_pgp','C_nadh']

params, C = make_symbols(param_names, conc_names)

display(v_6(params, C))

GapA*kI_adp_6*kI_amp_6*kI_atp_6*(C_g3p*C_nad*C_pi*kcat_f_6/(Km_g3p_6*Km_nad_6*Km_pi_6) - C_nadh*C_pgp*kcat_r_6/(Km_nadh_6*Km_pgp_6))/((C_adp + kI_adp_6)*(C_amp + kI_amp_6)*(C_atp + kI_atp_6)*((C_g3p/Km_g3p_6 + 1)*(C_nad/Km_nad_6 + 1)*(C_pi/Km_pi_6 + 1) + (C_nadh/Km_nadh_6 + 1)*(C_pgp/Km_pgp_6 + 1) - 1))

In [ ]:
def v_3(params, C, e):


    pass
def v_4(params, C, e):
    pass
def v_5(params, C, e):
    pass
def v_6(params, C, e):
    pass
def v_7(params, C, e):
    pass
def v_8(params, C, e):
    pass
def v_9(params, C, e):
    pass